# T-KGAT Stage 1 permutation control - all five seeds in one Kaggle run

Train the selected Stage 1 configuration with interaction timestamps shuffled globally
within train, using seeds 2019-2023. The selected gamma and other hyperparameters are
read from the completed Stage 1 recovery. Each seed starts its own model training.

Add the original code/data payload and the completed `tkgat-lastfm-stage1-results.zip`
as private Inputs. Enable GPU, then use **Save Version > Save & Run All**.
The default budget runs all five remaining fits in one background session.
The results ZIP is updated after every completed fit, including on a recoverable error.

In [ ]:
from pathlib import Path

PAYLOAD_DIR = None
# Set this only if automatic discovery finds multiple matching recoveries.
RESTORE_SOURCE = None
PROJECT_ROOT = Path('/kaggle/working/t-kgat-stage1-control')
RUN_TRAINING = True
NEW_CONTROL_FITS_THIS_SESSION = 5
CHECKPOINT_ZIP = Path('/kaggle/working/tkgat-lastfm-stage1-control-results.zip')

## 1. Restore completed Stage 1 results

For the first run, add `tkgat-lastfm-stage1-results.zip` with all 15 validation fits
and all five selected-model test results. Add the original `tkgat-lastfm-latest-kaggle.zip`
code/data payload too. Keep one recovery in Input, or set `RESTORE_SOURCE` explicitly.

For a later resume, use the newest `tkgat-lastfm-stage1-control-results.zip` as recovery.
Both ZIP files and Kaggle-extracted `results/` directories are supported.

In [ ]:
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile, is_zipfile


def recovery_archive(input_root, working, expected_identity, explicit=None):
    """Resolve one matching ZIP or extracted results directory, never guess between copies."""
    if explicit is not None:
        location = Path(explicit)
        if not location.exists():
            raise RuntimeError(f'Recovery path does not exist: {location}')
        locations = [location]
    else:
        locations = sorted(input_root.rglob('*.zip'))
        locations += sorted(p.parent for p in input_root.rglob('kaggle-run.json')
                            if p.parent.name == 'results')
    matches = []
    inspected = []
    for location in locations:
        if location.is_file() and is_zipfile(location):
            with ZipFile(location) as archive:
                marker = 'results/kaggle-run.json'
                if marker not in archive.namelist():
                    continue
                saved_identity = json.loads(archive.read(marker))
            candidate = ('zip', location)
        elif location.is_dir():
            folder = location if location.name == 'results' else location / 'results'
            marker = folder / 'kaggle-run.json'
            if not marker.is_file():
                continue
            saved_identity = json.loads(marker.read_text(encoding='utf-8'))
            candidate = ('directory', folder)
        else:
            continue
        inspected.append(str(location))
        if saved_identity == expected_identity:
            matches.append(candidate)
    if not matches:
        raise RuntimeError(
            'No matching Recovery found. Add the previous results as Input; '
            'both ZIP and extracted results/ are supported. '
            f'Found result locations: {inspected}. A completed Stage 1 recovery is required for its permutation control.'
        )
    if len(matches) != 1:
        raise RuntimeError(f'Multiple recoveries match: {matches}. Set RESTORE_SOURCE explicitly.')
    kind, source = matches[0]
    print('Recovery selected:', source, flush=True)
    if kind == 'zip':
        return source
    target = working / 'kgat-recovery-normalized.zip'
    with ZipFile(target, 'w', compression=ZIP_DEFLATED) as archive:
        for folder in (source, source.parent / 'reports'):
            if folder.is_dir():
                for path in sorted(folder.rglob('*')):
                    if path.is_file():
                        archive.write(path, path.relative_to(source.parent).as_posix())
    return target


def require_control_inputs(suite):
    """Read the selected Stage 1 configuration and validate its permutation control."""
    if not suite['complete']('tkgat'):
        raise RuntimeError('Stage 1 is incomplete in Recovery. Use its completed results ZIP.')
    print('tkgat: complete (verified)', flush=True)
    import yaml

    model_path = suite['ROOT'] / 'configs/model/tkgat_perm.yaml'
    config = yaml.safe_load(model_path.read_text(encoding='utf-8'))
    if (config.get('name') != 'tkgat' or config.get('time_on', 'interact') != 'interact'
            or config.get('control') != 'permute_train_ts' or config.get('from_tkgat') != 'tkgat'):
        raise RuntimeError('Expected the Stage 1 global interaction-timestamp permutation control.')
    if config.get('seeds') != list(range(2019, 2024)):
        raise RuntimeError('The control requires all five seeds 2019-2023.')
    runner = runpy.run_path(str(suite['ROOT'] / 'scripts/run.py'))
    folder = suite['ROOT'] / 'results/runs' / suite['RUN_NAME']
    points, selected = runner['tkgat_points'](config, model_path, folder)
    if len(points) != 1 or selected.get('gamma', 0) <= 0:
        raise RuntimeError('Expected one selected Stage 1 point with positive gamma.')
    print('Inherited Stage 1 configuration:', json.dumps(selected, sort_keys=True), flush=True)
    print('Control: permute_train_ts; seeds=[2019, 2020, 2021, 2022, 2023]; 5 fits.', flush=True)
    return points


input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
if PAYLOAD_DIR is None:
    manifests = sorted(input_root.rglob('payload-manifest.json'))
    if not manifests:
        packages = sorted(input_root.rglob('tkgat-lastfm-latest-kaggle.zip'))
        if len(packages) != 1:
            raise RuntimeError('Add the original code/data Dataset as Input, or set PAYLOAD_DIR.')
        unpacked = working / 'tkgat-payload'
        with ZipFile(packages[0]) as archive:
            for name in archive.namelist():
                path = PurePosixPath(name)
                if path.is_absolute() or '..' in path.parts or '\\' in name:
                    raise ValueError(f'Invalid payload member: {name}')
            archive.extractall(unpacked)
        manifests = [unpacked / 'payload-manifest.json']
    if len({path.read_text(encoding='utf-8') for path in manifests}) != 1:
        raise RuntimeError('Multiple code/data payloads found. Set PAYLOAD_DIR.')
    payload_root = manifests[0].parent
else:
    payload_root = Path(PAYLOAD_DIR)
manifest = json.loads((payload_root / 'payload-manifest.json').read_text(encoding='utf-8'))
bundle = payload_root / 'source.bundle'
if hashlib.sha256(bundle.read_bytes()).hexdigest() != manifest['files_sha256']['source.bundle']:
    raise ValueError('Source bundle hash mismatch')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', str(bundle), str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if commit != manifest['source_commit']:
    raise RuntimeError('Working copy belongs to another payload. Start a fresh session.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
suite = runpy.run_path(str(PROJECT_ROOT / 'scripts/kaggle_lastfm.py'))
suite['prepare'](payload_root)
has_progress = (PROJECT_ROOT / 'results/registry.csv').exists() or any(
    (PROJECT_ROOT / 'results/runs').rglob('*.pt')
)
if not has_progress:
    restore = recovery_archive(input_root, working, suite['identity'](), RESTORE_SOURCE)
    suite['extract_results'](restore)
else:
    print('Keeping existing working progress; Recovery will not overwrite it.', flush=True)
suite['record_identity']()
require_control_inputs(suite)
suite['backup'](CHECKPOINT_ZIP)
print('Source snapshot:', commit, flush=True)
print('Recovery verified. T-KGAT Stage 1 permutation control is the only model enabled.', flush=True)

## 2. Run all five control fits

Check that the log verifies `tkgat: complete (verified)` and displays the inherited
Stage 1 configuration. This notebook trains only `tkgat_perm.yaml`.

The five fits run sequentially. Completed fits are reused on resume, and a ZIP is
saved after each fit. An interrupted fit starts again from the beginning.
After all five validation fits, the runner evaluates the five control checkpoints
on test and writes `tkgat_perm_summary.json` with mean and sample standard deviation.

In [ ]:
import torch

require_control_inputs(suite)
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU T4 before Save & Run All.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
if NEW_CONTROL_FITS_THIS_SESSION < 1:
    raise ValueError('NEW_CONTROL_FITS_THIS_SESSION must be positive.')

if RUN_TRAINING:
    for step in range(NEW_CONTROL_FITS_THIS_SESSION):
        if suite['complete']('tkgat_perm'):
            print('tkgat_perm: complete. No other model will be started.', flush=True)
            break
        print(f'Running tkgat_perm Stage 1 control only; fit budget {step + 1}/{NEW_CONTROL_FITS_THIS_SESSION}', flush=True)
        try:
            suite['execute']([
                'scripts/run.py', '--data', suite['DATA_CONFIG'],
                '--model', 'configs/model/tkgat_perm.yaml', '--device', 'cuda',
                '--resume', '--max-new-runs', '1',
            ])
        finally:
            suite['backup'](CHECKPOINT_ZIP)
else:
    print('Training disabled. Recovery verified only.', flush=True)

suite['backup'](CHECKPOINT_ZIP)
print('tkgat_perm:', 'complete' if suite['complete']('tkgat_perm') else 'pending (more fits needed)', flush=True)
print('Save this output:', CHECKPOINT_ZIP, flush=True)

## 3. Download the final result

When the version finishes, download **`tkgat-lastfm-stage1-control-results.zip`**
from Output. Completion is confirmed by `tkgat_perm: complete` in the log.
The ZIP contains the inherited baselines and Stage 1 results together with the control.

If the run ends early, use its newest control ZIP as Recovery and run this notebook
again. Keep the original code/data Input. The five-fit budget covers the remaining
seeds and stops once the control is complete. Available GPU quota and the session
runtime limit still determine how long the background run can continue.